# Fulcher QC Plot Manual Check

Run this notebook from the repository root, or install the package in editable mode first. It renders synthetic QC examples for quick visual inspection without requiring archived SpectroCube data.

Compare the blend examples against `../reference_figures/r1fig6.png`: black dotted data with plus markers, compact legend, clear colored components, and hatched component ownership areas.

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "fulcher_extractor").exists()
)
sys.path.insert(0, str(repo_root / "src"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from fulcher_extractor.fit import FitConfig, LineFitResult, fit_line_group, fit_single_line
from fulcher_extractor.line_database import FulcherLine, load_lines
from fulcher_extractor.line_models import gaussian_area_model
from fulcher_extractor.qc import plot_line_fit, plot_region
from fulcher_extractor.spectrocube_io import Spectrum

preview_dir = repo_root / "local" / "qc_plot_manual_check"
preview_dir.mkdir(parents=True, exist_ok=True)
preview_dir

## Reviewer Reference

In [ ]:
display(Image(filename=str(repo_root / "examples" / "reference_figures" / "r1fig6.png")))

## Synthetic Spectrum

In [ ]:
sigma = 0.0273
line_a = FulcherLine("H2", "Q", 1, 1, "1-1", 9, 623.7457, "synthetic", "nm")
line_b = FulcherLine("H2", "Q", 2, 2, "2-2", 3, 623.8391, "synthetic", "nm")
line_iso = FulcherLine("H2", "Q", 0, 0, "0-0", 4, 605.6034, "synthetic", "nm")

wavelength = np.linspace(600.0, 630.0, 3001)
intensity = 0.12 + 0.015 * np.sin(wavelength * 9.0)
intensity += gaussian_area_model(wavelength, 0.08, line_a.wavelength_nm, sigma)
intensity += gaussian_area_model(wavelength, 0.03, line_b.wavelength_nm, sigma)
intensity += gaussian_area_model(wavelength, 0.045, line_iso.wavelength_nm, sigma)

spectrum = Spectrum(
    source_path="manual_check.py",
    shot_id="synthetic",
    selectors={"frame": 12},
    wavelength_nm=wavelength,
    intensity=intensity,
    intensity_units="a.u.",
    wavelength_medium="air",
    metadata={},
)

## Region Overview

Check that band rails replace full-height line forests, with only sparse Q labels.

In [ ]:
fig = plot_region(
    spectrum,
    wavelength_min_nm=600.0,
    wavelength_max_nm=630.0,
    lines=load_lines(),
    output_path=preview_dir / "region_overview.png",
)
plt.show()

## Isolated Fit

Check that `db` marks the database/rest line position and `fit` marks the fitted center. The summary note is drawn over the residual panel because the peak panel is the primary QC surface.

In [ ]:
config = FitConfig(instrument_sigma_nm=sigma, instrument_sigma_leeway_nm=0.005)
isolated = fit_single_line(spectrum, line_iso, config=config)
fig = plot_line_fit(
    spectrum,
    isolated,
    output_path=preview_dir / "isolated_fit.png",
)
plt.show()

## Two-Component Blend

Check that the assignment question is visually answerable: each assigned Fulcher component should have a colored curve, matching hatching, compact legend label, direct line label, and same-color `db`/`fit` position indicators.

In [ ]:
blend_results = fit_line_group(spectrum, [line_a, line_b], config=config)
blend_components = [
    {
        "label": result.line_id,
        "amplitude": result.amplitude,
        "center_nm": result.center_nm,
        "sigma_nm": result.sigma_nm,
        "rest_wavelength_nm": result.rest_wavelength_nm,
    }
    for result in blend_results
]
fig = plot_line_fit(
    spectrum,
    blend_results[0],
    components=blend_components,
    neighbor_lines=[line_a, line_b],
    output_path=preview_dir / "two_component_blend.png",
)
plt.show()

## Contamination Component

Check that a known non-reported contaminant can be plotted as a neutral grey `blended` Gaussian. The title should stay focused on the reported target, not the contaminant.

In [ ]:
target = FulcherLine("H2", "Q", 1, 1, "1-1", 4, 608.55, "synthetic", "nm")
contaminant_center = 608.64
contamination_wavelength = np.linspace(608.35, 608.85, 501)
contamination_intensity = 1.0
contamination_intensity += gaussian_area_model(contamination_wavelength, 0.06, target.wavelength_nm, sigma)
contamination_intensity += gaussian_area_model(contamination_wavelength, 0.025, contaminant_center, sigma)
contamination_spectrum = Spectrum(
    source_path="manual_check.py",
    shot_id="synthetic",
    selectors={"frame": 12},
    wavelength_nm=contamination_wavelength,
    intensity=contamination_intensity,
    intensity_units="a.u.",
    wavelength_medium="air",
    metadata={},
)
contamination_result = LineFitResult(
    line_id=target.line_id,
    isotopologue="H2",
    branch="Q",
    band=target.band,
    N=target.N,
    rest_wavelength_nm=target.wavelength_nm,
    amplitude=0.06,
    amplitude_stderr=0.001,
    center_nm=target.wavelength_nm,
    center_stderr_nm=0.0,
    sigma_nm=sigma,
    sigma_stderr_nm=0.0,
    fwhm_nm=2.354820045 * sigma,
    baseline_offset=1.0,
    baseline_strategy="local_minimum",
    window_min_nm=contamination_wavelength.min(),
    window_max_nm=contamination_wavelength.max(),
    background_min_nm=607.5,
    background_max_nm=609.5,
    n_points=contamination_wavelength.size,
    residual_rms=0.0,
    success=True,
    status="ok;known_contaminant",
    expected_center_nm=target.wavelength_nm,
    center_offset_from_rest_nm=0.0,
    center_offset_from_expected_nm=0.0,
    sigma_lower_bound_nm=0.015,
    sigma_upper_bound_nm=0.0423,
)
contamination_components = [
    {
        "label": contamination_result.line_id,
        "amplitude": contamination_result.amplitude,
        "center_nm": contamination_result.center_nm,
        "sigma_nm": contamination_result.sigma_nm,
        "rest_wavelength_nm": contamination_result.rest_wavelength_nm,
    },
    {
        "label": "contamination",
        "display_label": "blended",
        "role": "contaminant",
        "amplitude": 0.025,
        "center_nm": contaminant_center,
        "sigma_nm": sigma,
    },
]
fig = plot_line_fit(
    contamination_spectrum,
    contamination_result,
    components=contamination_components,
    output_path=preview_dir / "contamination_blended.png",
)
plt.show()

## Exact Coincidence

Check that exact database coincidences are explicitly labelled as unresolved rather than implying a spectral split.

In [ ]:
center = 626.2495
coincident_wavelength = np.linspace(626.05, 626.45, 401)
coincident_intensity = 1.0 + gaussian_area_model(coincident_wavelength, 0.08, center, sigma)
coincident_spectrum = Spectrum(
    source_path="manual_check.py",
    shot_id="synthetic",
    selectors={"frame": 12},
    wavelength_nm=coincident_wavelength,
    intensity=coincident_intensity,
    intensity_units="a.u.",
    wavelength_medium="air",
    metadata={},
)
coincident_result = LineFitResult(
    line_id="H2_Q10_1-1",
    isotopologue="H2",
    branch="Q",
    band="1-1",
    N=10,
    rest_wavelength_nm=center,
    amplitude=0.05,
    amplitude_stderr=0.001,
    center_nm=center,
    center_stderr_nm=0.0,
    sigma_nm=sigma,
    sigma_stderr_nm=0.0,
    fwhm_nm=2.354820045 * sigma,
    baseline_offset=1.0,
    baseline_strategy="local_minimum",
    window_min_nm=626.05,
    window_max_nm=626.45,
    background_min_nm=625.0,
    background_max_nm=627.5,
    n_points=coincident_wavelength.size,
    residual_rms=0.0,
    success=True,
    status="ok;blend_group;unresolved_coincident_database_lines",
    expected_center_nm=center,
    center_offset_from_rest_nm=0.0,
    center_offset_from_expected_nm=0.0,
    sigma_lower_bound_nm=0.015,
    sigma_upper_bound_nm=0.0423,
    center_lower_bound_nm=626.1,
    center_upper_bound_nm=626.33,
    blend_group_id="H2_Q10_1-1+H2_Q5_2-2",
    blend_component_count=2,
    close_neighbor_ids="H2_Q10_1-1,H2_Q5_2-2",
    blend_delta_nm=0.0,
)
coincident_components = [
    {"label": "H2_Q10_1-1", "amplitude": 0.05, "center_nm": center, "sigma_nm": sigma, "rest_wavelength_nm": center},
    {"label": "H2_Q5_2-2", "amplitude": 0.03, "center_nm": center, "sigma_nm": sigma, "rest_wavelength_nm": center},
]
fig = plot_line_fit(
    coincident_spectrum,
    coincident_result,
    components=coincident_components,
    output_path=preview_dir / "coincident_unresolved.png",
)
plt.show()

The rendered PNGs are written to `local/qc_plot_manual_check/`, which is ignored by git.